In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.data_quality_passed.bureau_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F

# ============================================================
# 1. READ DATA QUALITY PASSED BUREAU DATA
# ============================================================

source_df = spark.read.table(
    "fintech360.data_quality_passed.bureau_data"
)

# ============================================================
# 2. STANDARDIZE BUSINESS COLUMNS
# ============================================================

df = (source_df
    .withColumn("business_bureau_provider", F.upper(F.trim(F.col("bureau_provider"))))
    .withColumn("business_score_band", F.upper(F.trim(F.col("score_band"))))
    .withColumn("business_credit_score", F.col("credit_score").cast("int"))
    .withColumn("business_active_accounts", F.col("active_accounts").cast("int"))
    .withColumn("business_total_accounts", F.col("total_accounts").cast("int"))
    .withColumn("business_outstanding_amount", F.col("total_outstanding_amount").cast("double"))
    .withColumn("business_overdue_amount", F.col("overdue_amount").cast("double"))
    .withColumn("business_enquiries_30d", F.col("enquiries_last_30d").cast("int"))
    .withColumn("business_enquiries_90d", F.col("enquiries_last_90d").cast("int"))
    .withColumn("business_delinquency_ever", F.col("delinquency_ever").cast("int"))
    .withColumn("business_written_off_accounts", F.col("written_off_accounts").cast("int"))
    .withColumn("_report_ts", F.col("report_ts").cast("long"))
    .withColumn("_created_ts_ms", F.col("created_at").cast("long"))
    .withColumn("_updated_ts_ms", F.col("updated_at").cast("long"))
)

# ============================================================
# 3. CONVERT TIMESTAMPS
# report_ts is in Unix seconds.
# created_at and updated_at are in Unix milliseconds.
# ============================================================

df = (df
    .withColumn(
        "business_report_datetime",
        F.to_timestamp(F.from_unixtime(F.col("_report_ts")))
    )
    .withColumn(
        "business_report_date",
        F.to_date("business_report_datetime")
    )
    .withColumn(
        "business_report_month",
        F.date_format("business_report_datetime", "yyyy-MM")
    )
    .withColumn(
        "business_report_year",
        F.year("business_report_datetime")
    )
    .withColumn(
        "business_created_datetime",
        F.to_timestamp(
            F.from_unixtime(F.floor(F.col("_created_ts_ms") / 1000))
        )
    )
    .withColumn(
        "business_updated_datetime",
        F.to_timestamp(
            F.from_unixtime(F.floor(F.col("_updated_ts_ms") / 1000))
        )
    )
)

# ============================================================
# 4. CREDIT SCORE SEGMENTATION
# Illustrative score bands; confirm provider-specific thresholds.
# ============================================================

df = df.withColumn(
    "credit_score_segment",
    F.when(F.col("business_credit_score") < 600, "BELOW_600")
     .when(F.col("business_credit_score") < 650, "600_TO_649")
     .when(F.col("business_credit_score") < 700, "650_TO_699")
     .when(F.col("business_credit_score") < 750, "700_TO_749")
     .when(F.col("business_credit_score") < 800, "750_TO_799")
     .otherwise("800_PLUS")
)

# Preserve bureau-provided score_band separately.
df = df.withColumn(
    "bureau_reported_score_band",
    F.col("business_score_band")
)

# ============================================================
# 5. ACCOUNT PORTFOLIO FEATURES
# Active/total accounts ratio is not credit utilization.
# ============================================================

df = (df
    .withColumn(
        "inactive_accounts",
        F.greatest(
            F.col("business_total_accounts") -
            F.col("business_active_accounts"),
            F.lit(0)
        )
    )
    .withColumn(
        "active_account_ratio_pct",
        F.when(
            F.col("business_total_accounts") > 0,
            F.round(
                F.col("business_active_accounts") /
                F.col("business_total_accounts") * 100,
                2
            )
        )
    )
    .withColumn(
        "account_count_segment",
        F.when(F.col("business_total_accounts") == 0, "NO_REPORTED_ACCOUNTS")
         .when(F.col("business_total_accounts") <= 2, "1_TO_2_ACCOUNTS")
         .when(F.col("business_total_accounts") <= 5, "3_TO_5_ACCOUNTS")
         .otherwise("6_PLUS_ACCOUNTS")
    )
)

# ============================================================
# 6. DEBT EXPOSURE FEATURES
# ============================================================

df = (df
    .withColumn(
        "overdue_to_outstanding_pct",
        F.when(
            F.col("business_outstanding_amount") > 0,
            F.round(
                F.col("business_overdue_amount") /
                F.col("business_outstanding_amount") * 100,
                2
            )
        )
    )
    .withColumn(
        "outstanding_amount_band",
        F.when(F.col("business_outstanding_amount") < 10000, "BELOW_10K")
         .when(F.col("business_outstanding_amount") < 25000, "10K_TO_24999")
         .when(F.col("business_outstanding_amount") < 50000, "25K_TO_49999")
         .otherwise("50K_PLUS")
    )
    .withColumn(
        "overdue_amount_band",
        F.when(F.col("business_overdue_amount") == 0, "NO_OVERDUE")
         .when(F.col("business_overdue_amount") < 500, "BELOW_500")
         .when(F.col("business_overdue_amount") < 2000, "500_TO_1999")
         .when(F.col("business_overdue_amount") < 5000, "2000_TO_4999")
         .otherwise("5000_PLUS")
    )
    .withColumn(
        "has_outstanding_debt_flag",
        F.when(F.col("business_outstanding_amount") > 0, 1).otherwise(0)
    )
    .withColumn(
        "has_overdue_amount_flag",
        F.when(F.col("business_overdue_amount") > 0, 1).otherwise(0)
    )
)

# ============================================================
# 7. ENQUIRY INTENSITY
# ============================================================

df = (df
    .withColumn(
        "enquiries_31_to_90d",
        F.greatest(
            F.col("business_enquiries_90d") -
            F.col("business_enquiries_30d"),
            F.lit(0)
        )
    )
    .withColumn(
        "enquiry_intensity_30d",
        F.when(F.col("business_enquiries_30d") == 0, "NO_RECENT_ENQUIRIES")
         .when(F.col("business_enquiries_30d") <= 2, "LOW_ENQUIRY_ACTIVITY")
         .when(F.col("business_enquiries_30d") <= 5, "MODERATE_ENQUIRY_ACTIVITY")
         .otherwise("HIGH_ENQUIRY_ACTIVITY")
    )
    .withColumn(
        "enquiry_intensity_90d",
        F.when(F.col("business_enquiries_90d") == 0, "NO_ENQUIRIES")
         .when(F.col("business_enquiries_90d") <= 3, "LOW_ENQUIRY_ACTIVITY")
         .when(F.col("business_enquiries_90d") <= 6, "MODERATE_ENQUIRY_ACTIVITY")
         .otherwise("HIGH_ENQUIRY_ACTIVITY")
    )
)

# ============================================================
# 8. HISTORICAL DELINQUENCY AND WRITE-OFF SIGNALS
# ============================================================

df = (df
    .withColumn(
        "historical_delinquency_flag",
        F.when(F.col("business_delinquency_ever") == 1, 1).otherwise(0)
    )
    .withColumn(
        "written_off_account_flag",
        F.when(F.col("business_written_off_accounts") > 0, 1).otherwise(0)
    )
    .withColumn(
        "delinquency_review_signal",
        F.when(
            F.col("business_delinquency_ever") == 1,
            "HISTORICAL_DELINQUENCY_REPORTED"
        ).otherwise("NO_HISTORICAL_DELINQUENCY_REPORTED")
    )
)

# ============================================================
# 9. BUREAU REPORT RECENCY
# Uses a fixed reference timestamp for consistent calculations
# within this transformation run.
# ============================================================

reference_ts = F.current_timestamp()

df = (df
    .withColumn(
        "bureau_report_age_days",
        F.datediff(
            F.to_date(reference_ts),
            F.col("business_report_date")
        )
    )
    .withColumn(
        "bureau_report_recency_segment",
        F.when(F.col("bureau_report_age_days") < 0, "FUTURE_REPORT_DATE")
         .when(F.col("bureau_report_age_days") <= 30, "LAST_30_DAYS")
         .when(F.col("bureau_report_age_days") <= 90, "31_TO_90_DAYS")
         .when(F.col("bureau_report_age_days") <= 180, "91_TO_180_DAYS")
         .otherwise("OLDER_THAN_180_DAYS")
    )
)

# ============================================================
# 10. BUSINESS REVIEW SIGNALS
# These are indicators for investigation, not final risk decisions.
# ============================================================

df = df.withColumn(
    "bureau_review_signal",
    F.when(
        F.col("business_written_off_accounts") > 0,
        "WRITTEN_OFF_ACCOUNT_REVIEW"
    ).when(
        F.col("business_overdue_amount") > 0,
        "OVERDUE_EXPOSURE_REVIEW"
    ).when(
        F.col("business_delinquency_ever") == 1,
        "HISTORICAL_DELINQUENCY_REVIEW"
    ).when(
        F.col("business_enquiries_30d") >= 5,
        "HIGH_RECENT_ENQUIRY_REVIEW"
    ).otherwise("NO_BASIC_REVIEW_SIGNAL")
)

# ============================================================
# 11. TRANSFORMATION METADATA
# ============================================================

df = df.withColumn(
    "business_transformed_at",
    F.current_timestamp()
)

# ============================================================
# 12. FINAL BUSINESS DATAFRAME
# ============================================================

bureau_data_business = df

display(bureau_data_business.limit(10))


In [0]:

# Create destination schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.business_transformation
""")

# Append transformed bureau records
(bureau_data_business.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.business_transformation.bureau_data"))

print("Bureau business transformation completed.")
print("Target: fintech360.business_transformation.bureau_data")
